# Reviewer experiments: launch, leave running, refresh when wanted

Keep this notebook beside all supplied `.py` files. Runs are detached; no scrolling log output.

This suite uses a **new output folder** and leaves previous experiments untouched.
It includes all four experiments and the additional baseline/decomposition analyses.
Read `README.md` for the fixed selection rules and scientific limitations.

In [ ]:
from pathlib import Path
import sys, json, subprocess
import reviewer_suite as study

# Prefer the already-created Qwen3 environment; otherwise use this kernel's Python.
candidate = Path('/home/ubuntu/4/env_qwen3/bin/python')
WORKER_PYTHON = str(candidate) if candidate.exists() else sys.executable
check = subprocess.run([WORKER_PYTHON, '-c',
    "import json,torch,transformers; print(json.dumps(dict(torch=torch.__version__, transformers=transformers.__version__, cuda=torch.cuda.is_available())))"], capture_output=True, text=True)
if check.returncode:
    raise RuntimeError(check.stderr[-4000:])
print('Worker environment:', check.stdout.strip().splitlines()[-1])


## Settings — choose once before launch

The default below is an overnight starting batch: Pythia-410M, new seed 11,
both task families, all four experiments (36 conditions). Completion within
one night depends on measured speed; it pauses after 11.5 hours and resumes.
For the independent replication, subsequently run new seeds 12 and 13 as well.

To schedule the complete two-model, three-seed suite, remove the two reductions
marked below **before first launch** (216 conditions, potentially several nights).
Do not change scientific settings inside an existing output directory.

In [ ]:
SETTINGS = study.defaults(Path.cwd() / 'runs' / 'reviewer_overnight_seed11_v1')
SETTINGS['python'] = WORKER_PYTHON
SETTINGS['models'] = [m for m in SETTINGS['models'] if m['name'] == 'pythia410m']  # starting-batch reduction
SETTINGS['seeds'] = [11]  # starting-batch reduction; independent replication needs 12 and 13 too
SETTINGS['hours'] = 11.5
SETTINGS['minimum_free_gib'] = 40.
SETTINGS['max_output_gib'] = 180.

number = len(SETTINGS['models']) * len(SETTINGS['seeds']) * len(SETTINGS['tasks']) * len(study.conditions(SETTINGS))
print(f"Queued conditions: {number}; results: {SETTINGS['output']}")


## Launch / resume

Run this once. Re-running it while active simply returns the existing run.
The first launch pins model/dataset commits and may need internet access.
No package upgrades are performed.

In [ ]:
_ = study.launch(SETTINGS)


## Manual status refresh

Re-run this cell whenever you want. Refreshing does not modify the progress time.
`alive=True` means a worker holds the run lock; it does not guarantee recent progress.
A pause is resumable. A failure includes an error in this output and `worker.log`.

In [ ]:
print(json.dumps(study.status(SETTINGS), indent=2))


## Numbers and plots — only when requested

This reads the saved measurements. Incomplete branches are labelled. Compare
A retention with B learning; the natural-text and registry results stay separate.

In [ ]:
# Uncomment when you want the report and figures.
# study.show_results(SETTINGS)


## Stop or start a fresh run

Stop is cooperative. Wait until status says `alive=False`, then launch again
to resume. A fresh restart creates a different directory; it never deletes results.

In [ ]:
# study.stop(SETTINGS)
# After the worker has stopped, either run Launch/resume above or:
# SETTINGS = study.restart(SETTINGS)


## Share compact results

The ZIP includes a consistent SQLite snapshot, reports, provenance and code.
It intentionally omits heavyweight model/optimizer checkpoints.

In [ ]:
# ZIP_PATH = study.export(SETTINGS)
# print(ZIP_PATH)


## Optional: reanalyze existing results without new training

Accepts original four-vector/multimodel trajectory ZIPs or extracted folders.
Adds the stronger frozen-source baselines and the exact confusion/leakage
decomposition wherever recorded answer-mass measurements are available.

In [ ]:
# from analyze_existing import analyze_archive
# print(analyze_archive('/path/to/four_vector_share.zip', Path.cwd() / 'existing_reanalysis'))


## Optional: CPU smoke run

A tiny randomly initialized model verifies the pipeline; these are not paper results.
It uses a separate directory and takes about a minute on a laptop.

In [ ]:
# SMOKE = study.smoke_settings(Path.cwd() / 'runs' / 'reviewer_smoke')
# SMOKE['python'] = WORKER_PYTHON
# _ = study.launch(SMOKE)
# print(study.status(SMOKE))
